# HighDim-Inference-Toolkit Tutorial

This notebook demonstrates a minimal end-to-end workflow for:
- fitting Lasso via coordinate descent, and
- constructing a Wald confidence interval using Debiased Lasso.

If you installed from source in this repo, the imports below should work as-is.

In [ ]:
import numpy as np

from highdim_inference_toolkit.lasso import LassoCD
from highdim_inference_toolkit.debiased_lasso import DebiasedLasso
from highdim_inference_toolkit.utils import generate_high_dim_data

# Synthetic high-dimensional regression problem
X, y, beta_true = generate_high_dim_data(n=200, p=500, s=10, beta_strength=2.0, seed=0, sigma=1.0)

X.shape, y.shape, np.count_nonzero(beta_true)

In [ ]:
# Fit Lasso (point estimation)
lasso = LassoCD().fit(X, y, lambda_param=0.1, max_iter=5000, tol=1e-6)

# Fit Debiased Lasso (inference on individual coordinates)
db = DebiasedLasso(lambda_param=0.1, lambda_debias=0.1).fit(X, y)

j = 0
beta_db_j = db.debiased_coef(j)
ci_j = db.confidence_interval(j, alpha=0.05)

print(f"beta_true[{j}] = {beta_true[j]:.3f}")
print(f"lasso.coef_[{j}] = {lasso.coef_[j]:.3f}")
print(f"debiased[{j}] = {beta_db_j:.3f}")
print(f"95% CI = ({ci_j[0]:.3f}, {ci_j[1]:.3f})")

## Next steps

- Try different penalty levels (`lambda_param`, `lambda_debias`).
- Inspect multiple coordinates `j` and compare CIs against `beta_true`.
- Extend the notebook with transfer learning experiments using `highdim_inference_toolkit.trans_lasso.TransLasso`.

## Mini simulation: CI coverage (quick demo)

This is a lightweight Monte Carlo check that estimates empirical coverage of the 95% Wald CI from Debiased Lasso.
For a nicer plot + saved PNG, run `python scripts/make_coverage_figure.py` from the repo root.

In [ ]:
import numpy as np

from highdim_inference_toolkit.debiased_lasso import DebiasedLasso
from highdim_inference_toolkit.utils import generate_high_dim_data

def one_run(n, seed, p=80, s=5, beta_strength=3.0, sigma=1.0, j=0, lam=0.08):
    X, y, beta = generate_high_dim_data(n=n, p=p, s=s, beta_strength=beta_strength, seed=seed, sigma=sigma)
    m = DebiasedLasso(lambda_param=lam, lambda_debias=lam).fit(X, y)
    lo, hi = m.confidence_interval(j=j, alpha=0.05)
    return (lo <= beta[j] <= hi)

n_values = [120, 200, 350]
n_sims = 30
cover = []
for n in n_values:
    covered = sum(one_run(n=n, seed=2026 + 1000*n + t) for t in range(n_sims))
    cover.append(covered / n_sims)

list(zip(n_values, cover))